### This notebook can be used to calculate the interaction strength between features of GradientBoostingRegression model, using the H-statistic. This statistic
### quantifies the interaction strength between two features in a predictive model. The H-statistic ranges between 0 and 1, where 0 indicates no interaction
### and 1 indicates a strong interaction:

### 1.) read in the data, train the model using the setted target_var and optimized parameter settings
### 2.) calculate the H-statistic for pairs of features
### 3.) set a threshold to filter significant interactions



Load required moduls:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics
from numpy import mean
from numpy import std
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
import time
import os
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error

### create an output folder for interaction values:

In [2]:
# Define the directory path
dir_path = f"../../../output_data/ML_analysis/H_stat_interactions"

# Check if the directory exists, create it if it doesn't
if not os.path.exists(dir_path):
    os.makedirs(dir_path)


### Set target_var and model type: 

In [3]:
# select target_var: n_imbal, c_imbal, p_imbal, median_DIN, median_SRP or median_DOC_TOC_bioav
target_var = 'median_SRP'

#select model type: 'gbr' or 'lightgbm'
model_type = 'gbr'


Import dataset:

In [4]:
nutrient_export_data = pd.read_csv('../../../output_data/input_ML_learning/median_cnp_export_abs_conc_and_rfr_and_basin_feat.csv').drop(columns='Unnamed: 0')

Import optimized parameter sets:

In [5]:
optimized_parameter_sets = pd.read_csv('../../../output_data/ML_analysis/hyperparameter_settings/optimized_model_parms_for_importance_calculation.csv', sep = ';', dtype={'parameters': str})

# Convert the 'parameters' column to dictionaries
import ast
optimized_parameter_sets['parameters'] = optimized_parameter_sets['parameters'].apply(ast.literal_eval)

In [6]:
optimized_parameter_sets

,model,target,parameters,transformation
0,gbr,c_imbal,"{'learning_rate': 0.0018911001294483038, 'max_...",box-cox
1,gbr,n_imbal,"{'learning_rate': 0.001524207186070734, 'max_d...",box-cox
2,gbr,p_imbal,"{'learning_rate': 0.0018696900876637153, 'max_...",box-cox
3,gbr,median_DOC_TOC_bioav,"{'learning_rate': 0.0019202958248987234, 'max_...",box-cox
4,gbr,median_DIN,"{'learning_rate': 0.00420545212113823, 'max_de...",box-cox
5,gbr,median_SRP,"{'learning_rate': 0.0014628853574732998, 'max_...",box-cox
6,lightgbm,c_imbal,"{'colsample_bytree': 0.4, 'learning_rate': 0.0...",box-cox
7,lightgbm,n_imbal,"{'colsample_bytree': 0.4, 'learning_rate': 0.0...",box-cox
8,lightgbm,p_imbal,"{'colsample_bytree': 0.4, 'learning_rate': 0.0...",box-cox
9,lightgbm,median_DIN,"{'colsample_bytree': 0.4, 'learning_rate': 0.0...",box-cox


## Prepare the data for the machine learning models:

1. Split data into features and targets
2. Convert data frame to arrays 
3. Split data into training and test set

1. Function to prepare input data: split data into target and features and convert to arrays: 

In [7]:

def create_input_ML(data, target_var):

    # create array with exported nutrient values, which are our target values, we want to predict 
    target = np.array(nutrient_export_data[target_var])


    # remove the labels from the features df:

    features = nutrient_export_data.drop(['HYBAS_ID', 'n_imbal', 'p_imbal', 'c_imbal', 'median_DOC_TOC_bioav', 'median_DIN', 'median_SRP'], axis = 1)
    #features = nutrient_export_data.filter(items=selected_features)

    #select columns by index: 
    features= features.iloc[:,0:43]
    features.columns

    # save feature list for later use:
    feature_list = list(features.columns)

    # convert to numpy array:
    #features = np.array(features)

    return target, features, feature_list

### Apply create_input_ML function:

In [8]:
# data:
data = nutrient_export_data

In [9]:
target, features, feature_list = create_input_ML(data = data, target_var = target_var)

### Fit Boosting regression tree using best  parameters:

In [10]:
# extract best parameters: squeeze() funtion is required, to convert Series object to one value (result of loc operation is always a pandas series, even if output is only one value):
best_parms =  optimized_parameter_sets.loc[(optimized_parameter_sets['target'] == target_var) & (optimized_parameter_sets['model'] == model_type), 'parameters'].squeeze()

In [11]:
best_parms

{'learning_rate': 0.0014628853574732998,
 'max_depth': 10,
 'max_features': 4,
 'n_estimators': 6125,
 'subsample': 0.5}

Set random state values: use values from model optimization: 

In [12]:
# test size: usually 0.2 
test_size = 0.2
#set random_state:
random_state = 41 


In [13]:

# using Skicit-learn to split data into training and testing sets:
from sklearn.model_selection import train_test_split

train_features, test_features, train_targets, test_targets = train_test_split(features, target, test_size = test_size, random_state = random_state) 


Check data structure: targets and features should have same length in training and test set:

In [14]:
print('Training Features Shape:', train_features.shape)
print('Training targets Shape:', train_targets.shape)
print('Testing Features Shape:', test_features.shape)
print('Testing targets Shape:', test_targets.shape)

Training Features Shape: (2796, 33)
Training targets Shape: (2796,)
Testing Features Shape: (700, 33)
Testing targets Shape: (700,)


In [15]:
### Box-Cox transformation of training and testing targets:

In [16]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='box-cox')

# Reshape train_targets to 2D if it's 1D
if len(train_targets.shape) == 1:
    train_targets2 = np.reshape(train_targets, (-1, 1))

# Fit the transformer to the training data
# This learns the best lambda value for each feature
pt.fit(train_targets2)

# Transform the training data using the learned lambda values
train_targets_transformed_bc = pt.transform(train_targets2)
# convert back to 1D format
train_targets_transformed_bc = train_targets_transformed_bc.ravel()

# Transform the test data using the same lambda values
# Reshape test_targets to 2D if it's 1D
if len(test_targets.shape) == 1:
    test_targets2 = np.reshape(test_targets, (-1, 1))

test_targets_transformed_bc = pt.transform(test_targets2)
test_targets_transformed_bc = test_targets_transformed_bc.ravel()

### Now fit the model:

In [17]:

model = GradientBoostingRegressor(**best_parms,  random_state = 5, validation_fraction = 0.1, n_iter_no_change = 10)


# Fit the model:
model.fit(train_features, train_targets_transformed_bc)

# make predictions on train- and test-features:
pred_train = model.predict(train_features)
pred_test = model.predict(test_features)

r2_train = r2_score(train_targets_transformed_bc, pred_train)
r2_test = r2_score(test_targets_transformed_bc, pred_test)
print(f"R2 Train: {r2_train}")
print(f"R2 Test: {r2_test}")

R2 Train: 0.8224269287052007
R2 Test: 0.5132292585633033


### read in permutation importances for the corresponding model and target-variable:

In [18]:
relative_perm_importances = pd.read_csv(f'../../../output_data/ML_analysis/feature_importances/{target_var}_{model_type}/ranking_cutoff_{target_var}_most_rel_important_features.csv').drop(columns='Unnamed: 0')
most_important_features = relative_perm_importances[target_var].tolist()

In [19]:
most_important_features

['ppd_pk_uav', 'pre_mm_uyr', 'for_pc_use', 'crp_pc_use']

In [20]:
### Get indices of the most important features:

In [21]:
import itertools
# Liste der Spaltennamen
#most_important_features = ['ppd_pk_uav', 'wet_pc_ug2', 'pet_mm_uyr', 'slp_dg_uav', 'slt_pc_uav']

# Finde die Indizes der Spaltennamen in der Liste
col_indices = [train_features.columns.get_loc(col) for col in most_important_features]

# Calculate H-statistics for all pairs of features
feature_pairs = list(itertools.combinations(col_indices, 2))

In [22]:
# Liste der Spaltennamen
print(most_important_features)

# Finde die Indizes der Spaltennamen in der Liste
col_indices = [train_features.columns.get_loc(col) for col in most_important_features]
col_indices
feature_pairs = list(itertools.combinations(col_indices, 2))
print(feature_pairs)

feature_name_pairs = list(itertools.combinations(most_important_features, 2))
print(feature_name_pairs)

['ppd_pk_uav', 'pre_mm_uyr', 'for_pc_use', 'crp_pc_use']
[(6, 15), (6, 3), (6, 4), (15, 3), (15, 4), (3, 4)]
[('ppd_pk_uav', 'pre_mm_uyr'), ('ppd_pk_uav', 'for_pc_use'), ('ppd_pk_uav', 'crp_pc_use'), ('pre_mm_uyr', 'for_pc_use'), ('pre_mm_uyr', 'crp_pc_use'), ('for_pc_use', 'crp_pc_use')]


### Identify interactions according to Friedman H-statistics:

In [ ]:
import artemis

In [91]:
from artemis.interactions_methods.model_agnostic.partial_dependence_based import FriedmanHStatisticMethod

In [92]:
h_stat_method = FriedmanHStatisticMethod()

Now relative feature importances are selected:
read in file, which contains averaged most important feature ranking, up to the point where the next feature shows a clear cutoff¶


In [93]:
relative_perm_importances = pd.read_csv(f'../../../output_data/ML_analysis/feature_importances/{target_var}_{model_type}/ranking_cutoff_{target_var}_most_rel_important_features.csv').drop(columns='Unnamed: 0')

In [94]:
perm_imp_features = relative_perm_importances[f'{target_var}']
perm_imp_features = perm_imp_features.tolist()

In [95]:
perm_imp_features

['ppd_pk_uav',
 'pre_mm_uyr',
 'for_pc_use',
 'crp_pc_use',
 'slp_dg_uav',
 'gdp_ud_usu',
 'aet_mm_uyr']

In [96]:
### all samples: rename features:
all_samples = features

In [97]:
target_var

'median_SRP'

In [98]:
h_stat_method.fit(model=model, X=all_samples, features = perm_imp_features,show_progress=True, calculate_ova=False)

Calculating feature importance: 100%|███████████████████████████████████████████████| 33/33 [8:28:02<00:00, 923.72s/it]


In [99]:
interaction_values = h_stat_method.ovo  # One-versus-One Interaktionswerte
importance_values = h_stat_method.feature_importance  # Feature-Importanzwerte



In [100]:
interaction_values

,Feature 1,Feature 2,Friedman H-statistic Interaction Measure
0,for_pc_use,slp_dg_uav,0.022673
1,pre_mm_uyr,slp_dg_uav,0.012692
2,crp_pc_use,gdp_ud_usu,0.007190
3,pre_mm_uyr,for_pc_use,0.005039
4,slp_dg_uav,aet_mm_uyr,0.004595
5,ppd_pk_uav,for_pc_use,0.004376
6,ppd_pk_uav,crp_pc_use,0.003881
7,pre_mm_uyr,crp_pc_use,0.002848
8,gdp_ud_usu,aet_mm_uyr,0.002645
9,ppd_pk_uav,pre_mm_uyr,0.002145


In [30]:
# define the output folder path:

output_folder = f"{dir_path}/pyartemis_H_statistics"

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [101]:
interaction_values.to_csv(f"{dir_path}/pyartemis_H_statistics/H_statistics_Friedman_{target_var}.csv")

In [102]:
importance_values.to_csv(f"{dir_path}/pyartemis_H_statistics/pyartemis_importances_{target_var}.csv")

In [103]:
importance_values

,Feature,Importance
0,ppd_pk_uav,0.139588
1,slp_dg_uav,0.132627
2,for_pc_use,0.084978
3,pre_mm_uyr,0.084753
4,gdp_ud_usu,0.081149
5,crp_pc_use,0.080500
6,aet_mm_uyr,0.052617
